# General Database Connection

The first two cells import python libraries for handling data and database  
connections. The next cell establishes a connection to the ODOT data  
warehouse using your Windows credentials. It stores this connection in the 
conn object.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
connection_string = (
    r"Driver=SQL Server;"
    r"Server=DOTWarehousePrd;"
    r"Database=Warehouse;"
    r"Trusted_Connection=yes;"
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)
conn = engine.connect()

# Recreating the Original Excel Functionality

This is just something I typically do to make sure I understand the original  
source of a calculation, this just mirrors what was already being done in  
excel, but in python so that it can be incorporated in loops and more functional  
programming.

Skip to the "Data Selection Refinement" section for where I start doing  
new things.

## Recreating the "Initial Bridge List" Tab

In [3]:
# ---------------------------------------------------------
# 2. RAW SQL QUERY (REPLICATING ACCESS QUERY 1)
# ---------------------------------------------------------
# Note: Column names and order match the Access output for easy auditing.
sql_initial_bridge_list = text("""
SELECT DISTINCT 
    p.PROPOSAL_ID, 
    c.PROJECT_ID, 
    ps.PROPOSALSECTION_ID, 
    l.LETTING_DT, 
    l.LETTING_NM, 
    p.PROPOSAL_NM, 
    -- Mid([PROPOSAL_NM],4,Len([PROPOSAL_NM])-3) AS PID
    SUBSTRING(p.PROPOSAL_NM, 4, LEN(p.PROPOSAL_NM) - 3) AS PID, 
    p.CONTRACTALTERNATE_NM1, 
    p.AWARDEDAMOUNT, 
    p.FEDPROJECTNUM, 
    ps.PROPOSALSECTION_NM, 
    ps.DESCR AS [Section Desc], 
    c.ESSST3 AS [Existing SFN], 
    c.ESSST4 AS [New SFN], 
    c.FEDWORKCLASS, 
    bs.BRIDGE_NM, 
    bs.DESCR, 
    bs.TYPE, 
    bs.LENGTH, 
    bs.WIDTH, 
    bs.NUMOFSPANS, 
    'NO' AS Modified, 
    'NO' AS [Flag for Removal]
FROM dbo.PRECON_LETTING l
INNER JOIN dbo.PRECON_PROPOSAL p ON l.LETTING_ID = p.LETTING_ID
INNER JOIN dbo.PRECON_PROPOSALSECTION ps ON p.PROPOSAL_ID = ps.PROPOSAL_ID
INNER JOIN dbo.PRECON_PROJECTITEM pi ON ps.PROPOSALSECTION_ID = pi.PROPOSALSECTION_ID
INNER JOIN dbo.PRECON_CATEGORY c ON pi.CATEGORY_ID = c.CATEGORY_ID
-- Left join is used to ensure we keep bridges even if dimensions (bs) are missing
LEFT JOIN dbo.PRECON_BRIDGESEGMENT bs ON c.CATEGORY_ID = bs.CATEGORY_ID
WHERE 
    -- Letting Dates: FFY 2025 Cycle
    l.LETTING_DT >= '2024-10-01' AND l.LETTING_DT <= '2025-09-30'
    AND p.AWARDEDAMOUNT > 0 
    -- Work Class Filter (08-12)
    AND c.FEDWORKCLASS IN ('08', '09', '10', '11', '12')
    -- Category Filter (Structures 14/15)
    AND (LEFT(c.CATEGORY_NM, 2) = '14' OR LEFT(c.CATEGORY_NM, 2) = '15')
ORDER BY 
    l.LETTING_DT, 
    p.PROPOSAL_NM, 
    ps.PROPOSALSECTION_NM;
""")

In [4]:
# To see more than just a preview of the tables, uncomment this cell
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

In [5]:
with engine.connect() as conn:
    initial_bridge_list = pd.read_sql(sql_initial_bridge_list, conn)

initial_bridge_list.sort_values('LETTING_DT')

,PROPOSAL_ID,PROJECT_ID,PROPOSALSECTION_ID,LETTING_DT,LETTING_NM,PROPOSAL_NM,PID,CONTRACTALTERNATE_NM1,AWARDEDAMOUNT,FEDPROJECTNUM,...,New SFN,FEDWORKCLASS,BRIDGE_NM,DESCR,TYPE,LENGTH,WIDTH,NUMOFSPANS,Modified,Flag for Removal
0,9568.0,18200.0,103337.0,2024-10-17,24-10-17,FUL101140,101140,240464,3898020.95,E200(109),...,2601746,10,FUL-120-1408,STATE ROUTE 120 OVER TEN MILE CREEK,X030,36.37,60.3300,1.0,NO,NO
1,9582.0,18178.0,103433.0,2024-10-17,24-10-17,ROS116074,116074,244020,586885.00,E230(721),...,7104937,10,None,None,None,NaN,NaN,NaN,NO,NO
2,9642.0,18318.0,103906.0,2024-11-14,24-11-14,CLA102707,102707,240515,938819.94,E161(411),...,1202139,10,CLA-42-0820,STATE ROUTE 42 OVER GILROY DITCH.,X081,48.63,44.0000,1.0,NO,NO
3,9540.0,18165.0,103123.0,2024-11-14,24-11-14,CUY13184,13184,240510,4861698.83,E191(568),...,1801930,10,1801930,STATE ROUTE 4 OVER TINKERS CREEK,X081,119.67,45.0000,1.0,NO,NO
4,9540.0,18166.0,103123.0,2024-11-14,24-11-14,CUY13184,13184,240510,4861698.83,E191(568),...,1801930,10,1801930,STATE ROUTE 14 OVER TINKERS CREEK,X081,119.67,45.0000,1.0,NO,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,10212.0,19168.0,108532.0,2025-09-11,25-09-11,WOO107711,107711,250399,3693159.78,E210(245),...,None,10,None,None,None,NaN,NaN,NaN,NO,NO
80,10212.0,19169.0,108532.0,2025-09-11,25-09-11,WOO107711,107711,250399,3693159.78,E210(245),...,None,10,None,None,None,NaN,NaN,NaN,NO,NO
82,10252.0,19222.0,108889.0,2025-09-25,25-09-25,BRO114529,114529,250429,764901.05,E240(898),...,0802523,10,None,None,None,NaN,NaN,NaN,NO,NO
81,10263.0,19248.0,108974.0,2025-09-25,25-09-25,BRO100897,100897,250428,5107112.19,E170(246),...,0801120,10,BRO-52-12.452,US-52 OVER RED OAK CREEK.,X040,248.13,48.6667,3.0,NO,NO


## Working Backwards

Next Sheet after Initial Bridge List is "Bridge Types" in python, this is  
just a dictionary;

In [6]:
bridge_types = {
    "X0CB": "CABLE STAY BRIDGE OVER WATER",
    "X002": "TIMBER TRUSS BRIDGE",
    "X020": "CONCRETE SLAB OVER WATERWAY",
    "X028": "BOX CULVERT",
    "X030": "THREE SIDED CULVERT",
    "X031": "STEEL GIRDER BRIDGE OVER WATERWAY",
    "X032": "STEEL TRUSS BRIDGE OVER WATER",
    "X040": "STEEL BEAM BRIDGE OVER WATERWAY",
    "X081": "PRESTRESSED BRIDGE OVER WATERWAY",
    "X120": "CONCRETE SLAB BRIDGE OVER RAILROAD",
    "X131": "STEEL GIRDER BRIDGE OVER HIGHWAY",
    "X135": "STEEL BEAM BRIDGE OVER HIGHWAY",
    "X141": "STEEL BRIDGE OVER RAILROAD",
    "X150": "STEEL BEAM BRIDGE OVER RAILROAD",
    "X181": "PRESTRESSED BRIDGE OVER RAILROAD",
    "X220": "CONCRETE SLAB OVER HIGHWAY",
    "X281": "PRESTRESSED BRIDGE OVER HIGHWAY",
    "X320": "CONCRETE SLAB BRIDGE OVER WATER & RAIL",
    "X331": "STEEL GIRDER BRIDGE OVER WATER & RAIL",
    "X340": "STEEL BEAM BRIDGE OVER WATER AND RAIL",
    "X381": "PRESTRESSED BRIDGE OVER WATER& RAIL",
    "X420": "CONCRETE SLAB BRIDGE OVER WATER & HWY",
    "X481": "PRESTRESSED BRIDGE OVER WATER & HWY",
    "X520": "CONCRETE SLAB BRIDGE OVER RAIL & HWY",
    "X531": "STEEL GIRDER BRIDGE OVER RAIL & HWY",
    "X540": "STEEL BEAM BRIDGE OVER RAIL & HIGHWAY",
    "X581": "PRESTRESSED BRIDGE OVER RAIL & HWY",
    "X600": "STEEL GIRDER OVER WATER, RAIL & HWY",
    "X620": "STEEL BEAM BRIDGE OVER WATER, RAIL & HWY",
    "X700": "CONCRETE ARCH SECTIONS",
    "X999": "HIGHWAY TUNNEL",
    "Y007": "MINOR STRUCTURE (STORM SEWER, CULVERT)",
    "X033": "STEEL GIRDER BRIDGE OVER WATER & HWY",
}

## Federal Bridge Items FY25

In [10]:
import numpy as np
import pandas as pd
from sqlalchemy import text

# 1. Whitelist Filter: Get the 36 true bridges to avoid pulling items for noise/retaining walls
df_base = initial_bridge_list.copy()
df_base['Bridge Type Descr'] = df_base['TYPE'].map(bridge_types)
df_base = df_base[df_base['Bridge Type Descr'].notna()].drop_duplicates(['PROPOSAL_NM', 'Existing SFN'])
df_base['Bridge Area (Sq Ft)'] = pd.to_numeric(df_base['LENGTH'], errors='coerce').fillna(0) * pd.to_numeric(df_base['WIDTH'], errors='coerce').fillna(0)

# 2. Fetch Awarded Items for Bridge Categories only
sections = tuple(int(x) for x in df_base['PROPOSALSECTION_ID'].dropna().astype(float).unique())
sql = text(f"""
    SELECT DISTINCT pi.PROPOSALSECTION_ID, ri.ITEMCLASS, pi.PROPOSALITEM_LINENUM, ri.REFITEM_NM, ri.UNIT, 
           ri.DESCR AS [Item Description], pi.QTY, bid.BIDPRICE, bid.EXTENDEDAMOUNT AS Work_Line_Cost, v.VENDORNAME
    FROM dbo.PRECON_PROPOSALITEM pi
    INNER JOIN dbo.PRECON_BID bid ON pi.PROPOSALITEM_ID = bid.PROPOSALITEM_ID
    INNER JOIN dbo.PRECON_PROPOSALVENDOR pv ON bid.PROPOSALVENDOR_ID = pv.PROPOSALVENDOR_ID
    LEFT JOIN dbo.PRECON_REFVENDOR v ON pv.REFVENDOR_ID = v.REFVENDOR_ID
    LEFT JOIN dbo.PRECON_REFITEM ri ON pi.REFITEM_ID = ri.REFITEM_ID
    WHERE pv.AWARDED = 1 AND pi.PROPOSALSECTION_ID IN {str(sections)}
    AND EXISTS (
        SELECT 1 FROM dbo.PRECON_PROJECTITEM pji 
        INNER JOIN dbo.PRECON_CATEGORY c ON pji.CATEGORY_ID = c.CATEGORY_ID
        WHERE pji.PROPOSALITEM_ID = pi.PROPOSALITEM_ID
        AND c.FEDWORKCLASS IN ('08','09','10','11','12') AND (LEFT(c.CATEGORY_NM, 2) IN ('14','15'))
    )
""")

with engine.connect() as conn:
    df_items = pd.read_sql(sql, conn)

# 3. Clean ITEMCLASS & Vectorized Fed Cost Logic
def get_cls(r):
    v = str(r.get('ITEMCLASS', '')).strip().upper()
    if v and v not in ['NAN','NONE','']:
        p = v.split()
        return p[-1] if len(p) > 1 and p[0].isdigit() else v
    ref, desc = str(r.get('REFITEM_NM', '')), str(r.get('Item Description','')).upper()
    m = {'202':'RMVL', '203':'ERTH', '863':'ERTH', '507':'SPLS', '509':'SRBR', '511':'SCON', '526':'SCON', '513':'SSTL', '514':'BRPT', '601':'EC', '607':'FENC', '611':'DRNG', '613':'DRNG', '625':'LTNG', '840':'MISC', '867':'MISC'}
    return m.get(ref[:3], 'RMVL' if 'REMOV' in desc else 'SOTH')

df_items['ITEMCLASS'] = df_items.apply(get_cls, axis=1)
df_items['Fed_Work_Line_Cost'] = np.where(df_items['ITEMCLASS'].isin(['RMVL','EC','DRNG']), np.nan, df_items['Work_Line_Cost'])

# 4. Final Join and Format
df_base['PROPOSALSECTION_ID'] = df_base['PROPOSALSECTION_ID'].astype(float).astype(int).astype(str)
df_items['PROPOSALSECTION_ID'] = df_items['PROPOSALSECTION_ID'].astype(float).astype(int).astype(str)

federal_bridge_items = (pd.merge(df_base, df_items, on='PROPOSALSECTION_ID')
    .rename(columns={'AWARDEDAMOUNT':'Award Amt', 'DESCR':'Bridge Descr'})
    .assign(Comments='')
    .sort_values(['LETTING_NM', 'PROPOSAL_NM', 'PROPOSALITEM_LINENUM'])
    .reset_index(drop=True))

print(federal_bridge_items)

     PROPOSAL_ID  PROJECT_ID PROPOSALSECTION_ID LETTING_DT LETTING_NM  \
0       9,568.00   18,200.00             103337 2024-10-17   24-10-17   
1       9,568.00   18,200.00             103337 2024-10-17   24-10-17   
2       9,568.00   18,200.00             103337 2024-10-17   24-10-17   
3       9,568.00   18,200.00             103337 2024-10-17   24-10-17   
4       9,568.00   18,200.00             103337 2024-10-17   24-10-17   
..           ...         ...                ...        ...        ...   
979    10,263.00   19,248.00             108974 2025-09-25   25-09-25   
980    10,263.00   19,248.00             108974 2025-09-25   25-09-25   
981    10,263.00   19,248.00             108974 2025-09-25   25-09-25   
982    10,263.00   19,248.00             108974 2025-09-25   25-09-25   
983    10,263.00   19,248.00             108974 2025-09-25   25-09-25   

    PROPOSAL_NM     PID CONTRACTALTERNATE_NM1    Award Amt FEDPROJECTNUM  ...  \
0     FUL101140  101140                240

Seems to be working, but something is slightly off, returning a diff amount  
of the expected values, 984 here, 1025 in excel...

## Federal Bridge List FY25

In [8]:
# ---------------------------------------------------------
# 2. BASE DATA & CALCULATIONS
# ---------------------------------------------------------
# Use the initial_bridge_list generated from the first cell
df_list = initial_bridge_list.copy()

# Clean up SFNs early (replace string 'None' or 'nan' with actual blanks for cleaner Excel export)
df_list['Existing SFN'] = df_list['Existing SFN'].astype(str).str.strip()
df_list['Existing SFN'] = df_list['Existing SFN'].replace({'nan': '', 'None': '', 'none': '', '<NA>': ''})

# --- CRITICAL FIX: THE WHITELIST FILTER ---
# Tim's bridge_types dictionary isn't just for naming; it acts as a whitelist!
# By mapping the types and dropping NaNs, we automatically filter out retaining walls, 
# noise walls, and generic approach work that snuck into the Category 14/15 pull, 
# WHILE keeping the 3 valid bridges that happen to have 0 Area or missing SFNs.
df_list['Bridge Type Descr'] = df_list['TYPE'].map(bridge_types)
df_list = df_list[df_list['Bridge Type Descr'].notna()]

# Calculate Bridge Area BEFORE deduplicating so we can keep the row with actual dimensions
df_list['LENGTH'] = pd.to_numeric(df_list['LENGTH'], errors='coerce').fillna(0)
df_list['WIDTH'] = pd.to_numeric(df_list['WIDTH'], errors='coerce').fillna(0)
df_list['Bridge Area (Sq Ft)'] = df_list['LENGTH'] * df_list['WIDTH']

# Sort by Area (Descending) to ensure we keep the section with actual bridge dimensions
df_list = df_list.sort_values(by=['PROPOSAL_NM', 'Existing SFN', 'Bridge Area (Sq Ft)'], ascending=[True, True, False])

# Deduplicate strictly by Proposal and Existing SFN to get exactly one row per physical bridge.
df_list = df_list.drop_duplicates(subset=['PROPOSAL_NM', 'Existing SFN'])

# ---------------------------------------------------------
# 3. FETCH SMS BRIDGE DATA (NHS & DEFICIENCY)
# ---------------------------------------------------------
# Extract unique 'Existing SFN' values to query the SMS_BRIDGE table
sfns = [sfn for sfn in df_list['Existing SFN'].unique() if sfn != '']

# Format for SQL IN clause
if len(sfns) == 1:
    sfn_str = f"('{sfns[0]}')"
elif len(sfns) > 1:
    sfn_str = str(tuple(sfns))
else:
    sfn_str = "('')"

sql_sms = text(f"""
    SELECT 
        SFN,
        INVENT_NHS_CD,
        DEFIC_FUNC_RATING
    FROM dbo.SMS_BRIDGE
    WHERE SFN IN {sfn_str}
""")

try:
    with engine.connect() as conn:
        print("--- FETCHING SMS BRIDGE DATA ---")
        df_sms = pd.read_sql(sql_sms, conn)
        
    # Clean SFN in the SMS dataframe and drop any potential duplicates 
    # (in case SMS_BRIDGE retains historical records)
    df_sms['SFN'] = df_sms['SFN'].astype(str).str.strip()
    df_sms = df_sms.drop_duplicates(subset=['SFN'])

    # ---------------------------------------------------------
    # 4. MERGE AND FORMAT FINAL LIST
    # ---------------------------------------------------------
    # Merge the SMS data into the bridge list
    federal_bridge_list = pd.merge(
        df_list, 
        df_sms, 
        left_on='Existing SFN', 
        right_on='SFN', 
        how='left'
    )
    
    # Drop the duplicate SFN column from the merge
    if 'SFN' in federal_bridge_list.columns:
        federal_bridge_list = federal_bridge_list.drop(columns=['SFN'])

    # Define final column order based on the Excel screenshot
    final_cols = [
        'LETTING_DT', 'LETTING_NM', 'PROPOSAL_NM', 'PID', 'CONTRACTALTERNATE_NM1',
        'AWARDEDAMOUNT', 'FEDPROJECTNUM', 'PROPOSALSECTION_NM', 'Section Desc',
        'Existing SFN', 'New SFN', 'FEDWORKCLASS', 'BRIDGE_NM', 'DESCR', 'TYPE',
        'LENGTH', 'WIDTH', 'NUMOFSPANS', 'Modified', 'Flag for Removal',
        'Bridge Type Descr', 'Bridge Area (Sq Ft)', 'INVENT_NHS_CD', 'DEFIC_FUNC_RATING'
    ]

    # Ensure all columns exist, fill missing with blank
    for col in final_cols:
        if col not in federal_bridge_list.columns:
            federal_bridge_list[col] = ''

    federal_bridge_list = federal_bridge_list[final_cols]
    
    # Sort for review
    federal_bridge_list = federal_bridge_list.sort_values(['LETTING_DT', 'PROPOSAL_NM'])

    print(f"--- FEDERAL BRIDGE LIST GENERATED ---")
    print(f"Total Bridges: {len(federal_bridge_list)}")
    print("\nPreview of the Federal Bridge List:")
    pd.options.display.float_format = '{:,.2f}'.format
    print(federal_bridge_list[['PROPOSAL_NM', 'Existing SFN', 'Bridge Area (Sq Ft)', 'INVENT_NHS_CD', 'DEFIC_FUNC_RATING']].head(15))

except Exception as e:
    print(f"Error processing SMS Bridge data: {e}")

--- FETCHING SMS BRIDGE DATA ---
--- FEDERAL BRIDGE LIST GENERATED ---
Total Bridges: 36

Preview of the Federal Bridge List:
   PROPOSAL_NM Existing SFN  Bridge Area (Sq Ft) INVENT_NHS_CD  \
15   FUL101140      2601745             2,194.20             0   
4    CLA102707      1202138             2,139.72             0   
8     CUY13184      1801929             5,385.15             0   
17   GEA111000      2800756             3,237.38             0   
18   GEA115823      2801507             3,629.32             0   
24   LOG114937      4602315             1,950.12             0   
35   WAR112975      8305161             6,452.02             0   
31   PIK106540      6630839             2,464.00             0   
10   DEL105433      2103575             2,320.00             0   
11   DEL105433      2103664             2,312.00             0   
19   HAM102790      3106632             4,962.48             1   
22   LIC104981      4501810             4,140.80             0   
23   LIC104981  

In [9]:
# ==============================================================================
# OHIO DOT BRIDGE REPLACEMENT - BRIDGE COST TOTALS
# Objective: Recreate the "Bridge Costs Totals" sheet by merging the clean 
# Federal Bridge List with the aggregated costs from the Federal Bridge Items.
# ==============================================================================

import pandas as pd

# ---------------------------------------------------------
# 1. AGGREGATE ITEMIZED COSTS
# ---------------------------------------------------------
# Use the federal_bridge_items dataframe generated in Step 2.
# We group by Proposal and Bridge Name to sum the costs.
# Note: pandas .sum() automatically treats the NaNs in Fed_Work_Line_Cost as 0!

df_items = federal_bridge_items.copy()

# Ensure numeric types before summing
df_items['Work_Line_Cost'] = pd.to_numeric(df_items['Work_Line_Cost'], errors='coerce').fillna(0)
df_items['Fed_Work_Line_Cost'] = pd.to_numeric(df_items['Fed_Work_Line_Cost'], errors='coerce').fillna(0)

# Aggregate the costs
cost_totals = df_items.groupby(['PROPOSAL_NM', 'BRIDGE_NM']).agg({
    'Work_Line_Cost': 'sum',
    'Fed_Work_Line_Cost': 'sum'
}).reset_index()

# Rename columns to match the Pivot Table output
cost_totals = cost_totals.rename(columns={
    'Work_Line_Cost': 'Sum of Work_Line_Cost',
    'Fed_Work_Line_Cost': 'Sum of Fed_Work_Line_Cost'
})

# ---------------------------------------------------------
# 2. MERGE COSTS ONTO THE BRIDGE LIST
# ---------------------------------------------------------
# Use the deduplicated federal_bridge_list (36 bridges) generated in Step 3.
df_list = federal_bridge_list.copy()

# Merge the aggregated costs onto the bridge list
final_totals = pd.merge(
    df_list,
    cost_totals,
    on=['PROPOSAL_NM', 'BRIDGE_NM'],
    how='left'
)

# Fill any missing costs with 0 just in case a bridge had no billable items
final_totals['Sum of Work_Line_Cost'] = final_totals['Sum of Work_Line_Cost'].fillna(0)
final_totals['Sum of Fed_Work_Line_Cost'] = final_totals['Sum of Fed_Work_Line_Cost'].fillna(0)

# ---------------------------------------------------------
# 3. CALCULATE FINAL METRICS
# ---------------------------------------------------------
# Calculate Unit Cost (Cost per Sq Ft) using the FEDERAL work line cost.
# We use a lambda function to safely handle division by zero for any 
# rare bridges that might have 0 Area.
final_totals['Cost per Sq Ft'] = final_totals.apply(
    lambda row: row['Sum of Fed_Work_Line_Cost'] / row['Bridge Area (Sq Ft)'] if row['Bridge Area (Sq Ft)'] > 0 else 0,
    axis=1
)

# Select and order the final columns exactly as Tim reviews them
final_columns = [
    'LETTING_DT', 'LETTING_NM', 'PROPOSAL_NM', 'PID', 'CONTRACTALTERNATE_NM1',
    'AWARDEDAMOUNT', 'FEDPROJECTNUM', 'PROPOSALSECTION_NM', 'Section Desc',
    'Existing SFN', 'New SFN', 'FEDWORKCLASS', 'BRIDGE_NM', 'DESCR', 'TYPE',
    'LENGTH', 'WIDTH', 'NUMOFSPANS', 'Modified', 'Flag for Removal',
    'Bridge Type Descr', 'Bridge Area (Sq Ft)', 'INVENT_NHS_CD', 'DEFIC_FUNC_RATING',
    'Sum of Work_Line_Cost', 'Sum of Fed_Work_Line_Cost', 'Cost per Sq Ft'
]

# Ensure all columns exist to prevent KeyError
for col in final_columns:
    if col not in final_totals.columns:
        final_totals[col] = ''

final_totals = final_totals[final_columns]

# Sort alphabetically by Proposal and Bridge Name for Tim's review
final_totals = final_totals.sort_values(['PROPOSAL_NM', 'BRIDGE_NM'])

# ---------------------------------------------------------
# 4. OUTPUT PREVIEW & EXPORT
# ---------------------------------------------------------
pd.options.display.float_format = '{:,.2f}'.format

print(f"--- FINAL BRIDGE COST TOTALS REPLICATED ---")
print(f"Total Unique Bridges: {len(final_totals)}")

print("\nPreview of the Final Pivot Table:")
print("Notice the difference between Work Cost and Fed Work Cost where items were excluded!\n")
print(final_totals[['PROPOSAL_NM', 'BRIDGE_NM', 'Bridge Area (Sq Ft)', 'Sum of Work_Line_Cost', 'Sum of Fed_Work_Line_Cost', 'Cost per Sq Ft']].head(15))

# Uncomment this line if you want to export the final result straight to Excel!
# final_totals.to_excel('Bridge_Cost_Totals_FY25.xlsx', index=False)

--- FINAL BRIDGE COST TOTALS REPLICATED ---
Total Unique Bridges: 36

Preview of the Final Pivot Table:
Notice the difference between Work Cost and Fed Work Cost where items were excluded!

   PROPOSAL_NM         BRIDGE_NM  Bridge Area (Sq Ft)  Sum of Work_Line_Cost  \
33   ALL117742       SFN 0203484             1,444.00             572,398.20   
32   ATH119810   ATH-MADSN-00400             2,996.79           1,552,937.50   
30   BEL117373  BEL-C0004-27.460             2,490.00           1,012,554.80   
35   BRO100897     BRO-52-12.452            12,075.67           4,162,897.77   
1    CLA102707       CLA-42-0820             2,139.72             637,184.04   
34   CLI105224      CLI-134-0225             4,586.80           1,199,975.20   
21   CUY104132       CUY-14-0693            34,396.54          21,591,451.80   
22   CUY104132    CUY-CR240-0061             3,360.00           2,744,377.01   
2     CUY13184           1801930             5,385.15           1,059,855.00   
18   DEF11

## Conclusion on Excel Portion

There are minor discrepencies above, likely the result from values being  
manually filtered out during the fine grained review process.



# Data Selection Refinement

## Review of FHWA Requirements

### Unit Cost Criteria

FHWA has their [Unit Cost criteria](https://www.fhwa.dot.gov/bridge/nbi/uc_criteria.pdf).

#### 1. Deadline and Formula

> Unit costs for the replacement of all highway bridges constructed with  
Federal funds on the NHS and off the NHS are to be submitted by April 1 of  
each year. The total cost of eligible items is divided by the total deck  
area of the new replacement bridges to determine the average unit cost by  
system.

Straight forward, deck area is being manually calculated, better to take  
from AssetWise long term.

#### 2. Scope

> All replaced highway bridges let or awarded during the appropriate  
fiscal year are to be used. Please indicate the number of bridges and area  
used to calculate the unit costs for replacement for each system.

This is already starting to confuse me. This says let/award, but project  
closeout is what actually matters right? Like if a project gets let and  
awarded in 2023, could it be closed out in 2025? How would that work?

> //TODO - determine whether to filter queries by letting, award, or closeout date

#### 3. Scope Cont.

> Exclude culverts (multiple cell box culverts, long span culverts and  
multiple pipe installations) from the calculations. The NBI definition and  
the coding of Item 43 of the Recording and Coding Guide for the Structure  
Inventory and Appraisal of the Nation's Bridges (Coding Guide) should be  
used to distinguish culverts from bridges.

Not much to say other than switch from NBI 43 to B.SP.01 from AssetWise?

Don't know if it's being pulled into SMS yet, //TODO - Check

#### 4. Deck Area

> The total deck area of the replacement bridge is to be used for all  
calculations. The length dimension is to be as described for Item 49 and  
the width dimension is to be as described for Item 52 in the Coding Guide.  
These data are to be consistent with the information submitted for the  
particular bridge in the NBI.

//TODO - This has changed a bit with SNBI, I doubt SMS is incorporating the  
changes. We would probably just replace with B.G.16 - Calculated Deck Area  
but might 

#### 5. Exceptional Bridges

> Bridges involving unusual circumstances or types of construction not  
routinely used by the State that significantly raise or lower the unit cost  
should not be included. Generally, certain types of bridges can be  
identified as unusual - movable, cable-stayed, suspension, segmental, and  
other structures that have a clear unsupported length greater than 500  
feet. However, certain States that have built numerous types of unusual  
bridges, e.g. segmental bridges, may no longer consider such bridges as  
unusual. Therefore, the States are given some discretion in making the  
determination concerning the definition of unusual. Unusual circumstances  
may include extremely difficult access conditions and the occurrence of  
extreme events during construction.

//TODO - I honestly have no idea what this means. Their wording makes it  
seems like we could completely ignore this and be fine, other than like  
cable-stays or suspension bridge types, so I guess just throw in an  
additional B.SP.01 filter?

#### 6. Staged Construction

> Bridges that are under staged construction should not be included unless  
the final stage has been bid and a total unit cost can be obtained. If a  
bridge is included in a design-build or lump sum contract, if possible,  
please determine the eligible costs for inclusion in the unit cost  
calculations. If this cannot be accomplished, the State should exclude the  
bridge from the unit cost submittal.

//TODO - This is the one I don't understand and kind of conflicts with #2.  
How often does this occur and if it's a staged construction for like Brent  
Spence or something, do we have to go back in later years and bring those  
costs forward?

#### 7. Excluded Items

> Unit costs shall be based on bridge costs only. A list of specific items  
to be excluded is provided below. The list is not all-inclusive and care  
should be taken to ensure that other similar items are also excluded.

//TODO - Determine white/blacklist filter for item numbers if more practical

#### 8. Rounding

> Unit costs shall be rounded up to the next highest dollar.

### **Items to be Excluded**

#### 1. Mobilization

Item 624

#### 2. Demolition of Existing Bridges

Item 202

#### 3. Approach Slabs (approach slabs may be included when paid for as bridge item, e.g. on integral abutment bridge).

#### 4. Stream Channel Work, Riprap, Slope Paving

601 (Riprap), 659 (Seeding), 670 (Slope Protection)

#### 5. Earthwork (exclusive of structural excavation, structural backfill, and earthwork associated with Geosynthetic Reinforced Soil Integrated Bridge Systems)

Item 203

#### 6. Clearing and Grubbing

#### 7. Retaining Walls not attached to Abutments

#### 8. Guardrail Transitions to Bridges

#### 9. Maintenance and Protection of Traffic

#### 10. Detour Costs

#### 11. Signing and Marking

#### 12. Lighting

#### 13. Electrical Conduits

#### 14. Inlet Frames and Grates

#### 15. Field Office

Item 619

#### 16. Construction Engineering Items

#### 17. Training

#### 18. Right-of-Way

#### 19. Utility Relocation

#### 20. Contingencies

# Data Mapping and Investigation

## SMS Bridge

Most of the bridge feature information is coming from here. But I'm pretty  
sure it's a historical data resource pulling modern values from AssetWise.

In [22]:
import pandas as pd
from sqlalchemy import text

def safe_profile_table(table_name, engine):
    print(f"\n{'='*80}\nSAFE SCHEMA PROFILE: {table_name}\n{'='*80}")
    
    # 1. Get Column Metadata (Standard Information Schema - very safe for older drivers)
    meta_query = text(f"""
        SELECT 
            COLUMN_NAME, 
            DATA_TYPE, 
            IS_NULLABLE,
            COLUMN_DEFAULT
        FROM INFORMATION_SCHEMA.COLUMNS 
        WHERE TABLE_NAME = '{table_name}'
        ORDER BY ORDINAL_POSITION
    """)
    
    # 2. Get Primary Key Status (Standard Constraint Schema)
    pk_query = text(f"""
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE OBJECTPROPERTY(OBJECT_ID(CONSTRAINT_SCHEMA + '.' + CONSTRAINT_NAME), 'IsPrimaryKey') = 1
        AND TABLE_NAME = '{table_name}'
    """)
    
    # 3. Get Data Sample
    sample_query = text(f"SELECT TOP 500 * FROM dbo.{table_name}")
    
    with engine.connect() as conn:
        df_meta = pd.read_sql(meta_query, conn)
        df_pks = pd.read_sql(pk_query, conn)
        df_sample = pd.read_sql(sample_query, conn)
    
    pk_list = df_pks['COLUMN_NAME'].tolist()
    summary_rows = []
    
    for _, row in df_meta.iterrows():
        col = row['COLUMN_NAME']
        tag = "(PK)" if col in pk_list else ""
        
        # Calculate Usage %
        usage = (df_sample[col].count() / len(df_sample)) * 100 if not df_sample.empty else 0
        sample_val = df_sample[col].dropna().iloc[0] if not df_sample[col].dropna().empty else "NULL"
        
        summary_rows.append({
            "Column Name": f"{col} {tag}".strip(),
            "Type": row['DATA_TYPE'],
            "Nullable": row['IS_NULLABLE'],
            "Usage %": f"{usage:.1f}%",
            "Sample Data": str(sample_val)[:50]
        })
    
    print(pd.DataFrame(summary_rows).to_string(index=False))

# Run for your primary data source
safe_profile_table('SMS_BRIDGE', engine)


SAFE SCHEMA PROFILE: SMS_BRIDGE
            Column Name     Type Nullable Usage %         Sample Data
                    SFN  varchar       NO  100.0%             0933244
              COUNTY_CD  varchar      YES  100.0%                 BUT
                  ROUTE  varchar      YES  100.0%               C0215
                    SLM  varchar      YES  100.0%               02000
                    SLK  varchar      YES  100.0%               00322
               DISTRICT  varchar      YES  100.0%                  08
                BARS_CD  varchar      YES  100.0%                   0
                FIPS_CD  varchar      YES  100.0%               52080
       INVENT_ON_UND_CD  varchar      YES  100.0%                   1
      INVENT_HWY_SYS_CD  varchar      YES  100.0%                   4
           INVENT_ROUTE  varchar      YES  100.0%               C0215
      INVENT_DIR_SFX_CD  varchar      YES  100.0%                   0
     INVENT_HWY_DSGT_CD  varchar      YES  100.0%        

Only two values are really being pulled from here, length and width,  
everything else is basically AASHTO Values.

## AASHTOWare PreConstruction Data

Is this mostly precon stuff? It's all phases right? most of the tables are  
labeled as precon.

In [23]:
# Target the core AASHTO tables needed for the Item/Cost logic
precon_targets = [
    'PRECON_PROPOSALITEM',  # Has the item numbers (e.g., 202, 624)
    'PRECON_BID',           # Has the money and Awarded flag
    'PRECON_CATEGORY'       # Has the 14/15 bridge category flags
]

for table in precon_targets:
    try:
        safe_profile_table(table, engine)
    except Exception as e:
        print(f"Could not summarize {table}: {e}")


SAFE SCHEMA PROFILE: PRECON_PROPOSALITEM
Could not summarize PRECON_PROPOSALITEM: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC SQL Server Driver][SQL Server]The SELECT permission was denied on the column 'UNITPRICE' of the object 'PRECON_PROPOSALITEM', database 'Warehouse', schema 'dbo'. (230) (SQLExecDirectW); [42000] [Microsoft][ODBC SQL Server Driver][SQL Server]The SELECT permission was denied on the column 'EXTENDEDAMOUNT' of the object 'PRECON_PROPOSALITEM', database 'Warehouse', schema 'dbo'. (230)")
[SQL: SELECT TOP 500 * FROM dbo.PRECON_PROPOSALITEM]
(Background on this error at: https://sqlalche.me/e/20/f405)

SAFE SCHEMA PROFILE: PRECON_BID
             Column Name     Type Nullable Usage %         Sample Data
                  BID_ID  numeric      YES  100.0%            172759.0
       PROPOSALVENDOR_ID  numeric      YES  100.0%              9306.0
    PROPOSALITEM_LINENUM  varchar      YES  100.0%                0108
         PROPOSALITEM_ID  numeric     